<h1>Parallel to fine tune</h1>

In [2]:
#!/usr/bin/env python3
"""
Zero-shot GLiNER2.5 baseline for hierarchical noun semantic tagging.

Pipeline
--------
1. Run native GLiNER2.5 span extraction over UD English EWT training documents
   with a deliberately low raw extraction threshold.
2. Match extracted spans back to NOUN/PROPN tokens.
3. Use a held-out validation subset of the TRAINING corpus to optimise one
   final acceptance threshold per directly extractable semantic feature.
4. Freeze those thresholds.
5. Run the same standard GLiNER2.5 extraction pipeline over the official EWT
   test corpus and apply the frozen thresholds.
6. Add deterministic hierarchy inheritance and write one JSON object per
   NOUN/PROPN token.
7. Evaluate on the official held-out test annotations only after threshold
   calibration is frozen, including per-feature tables and a sentence-cluster
   bootstrap matching the RoBERTa reporting design.

No GLiNER weights are fine-tuned. Gold training annotations are used only for
sentence-level split construction and threshold calibration. Official test gold
annotations are read only after every validation-derived decision is frozen.

The script deliberately uses the native GLiNER2.5 entity/span extraction API;
it does not classify one preselected noun at a time.

Python syntax is compatible with Python 3.10, although current GLiNER2
releases themselves may require a newer Python runtime.
"""

from __future__ import annotations

import hashlib
import json
import random
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import torch
from tqdm import tqdm
from gliner2 import AutoExtractor
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


# =============================================================================
# Configuration
# =============================================================================

TRAIN_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-train.conllu"
)

TEST_CONLLU_PATH = Path(
    "../data/UD_English-EWT-master/"
    "UD_English-EWT-master/"
    "en_ewt-ud-test.conllu"
)

# Gold semantic annotations produced by the other tagging pipeline.
# These are used ONLY to tune thresholds on a held-out subset of the
# TRAINING corpus. Official test annotations are read only after threshold
# calibration is frozen, solely for final evaluation and bootstrapping.
GOLD_TRAIN_JSONL_PATH = Path(
    "../data/ewt_train_wordnet_wsd_features_wordnet_fixed.jsonl"
)

# Official held-out gold test annotations. These are read ONLY after all
# validation threshold calibration is complete, and are used only for final
# evaluation/bootstrapping.
GOLD_TEST_JSONL_PATH = Path(
    "../data/ewt_test_wordnet_wsd_features_wordnet_fixed.jsonl"
)

# Canonical, model-independent sentence split used for direct system
# comparisons. The GLiNER script validates this split against the same eligible
# sentence set used by RoBERTa. If it does not exist, the split is reconstructed
# deterministically with the same sentence-level multilabel stratification
# design as the RoBERTa experiment and saved here for reuse by BOTH systems.
SHARED_SPLIT_PATH = Path(
    "../data/semantic_feature_sentence_split.json"
)

# Legacy RoBERTa output location. A split from here is imported only if it
# exactly covers the current eligible sentence set and has the expected 20%
# validation size. This prevents an older/stale split (such as 8,885/2,313)
# from silently contaminating the comparison.
ROBERTA_SPLIT_PATH = Path(
    "../models/roberta_semantic_features/sentence_split.json"
)

# GLiNER keeps a copy of the canonical split for provenance.
GLINER_SPLIT_PATH = Path(
    "../models/gliner2.5/gliner2_validation_split.json"
)

THRESHOLDS_PATH = Path(
    "../models/gliner2.5/gliner2_validation_thresholds.json"
)

TRAIN_RAW_SCORES_PATH = Path(
    "../models/gliner2.5/ewt_train_gliner2_raw_scores.jsonl"
)

TEST_RAW_SCORES_PATH = Path(
    "../models/gliner2.5/wt_test_gliner2_raw_scores.jsonl"
)

TEST_OUTPUT_PATH = Path(
    "../models/gliner2.5/ewt_test_gliner2_noun_features_span.jsonl"
)

TEST_RESULTS_PATH = Path(
    "../models/gliner2.5/gliner2_test_results.json"
)

BOOTSTRAP_RESULTS_PATH = Path(
    "../models/gliner2.5/gliner2_test_bootstrap_confidence_intervals.json"
)

TRAIN_UNMATCHED_SPANS_PATH = Path(
    "../models/gliner2.5/ewt_train_gliner2_unmatched_spans.jsonl"
)

TEST_UNMATCHED_SPANS_PATH = Path(
    "../models/gliner2.5/ewt_test_gliner2_unmatched_spans.jsonl"
)

MODEL_NAME = "fastino/gliner2.5-base-v1"

TARGET_POS = {
    "NOUN",
    "PROPN",
}

# Validation is used only for threshold calibration.
VALIDATION_FRACTION = 0.20
RANDOM_SEED = 42

# These match the RoBERTa experiment's sentence-level split construction.
SPLIT_SEED = 42
MIN_TRAIN_POSITIVES_PER_FEATURE = 1
MAX_STRATIFICATION_ATTEMPTS = 200

# Match the other evaluation pipeline: gold NOUN/PROPN records with no semantic
# features are not treated as valid all-negative calibration examples.
EXCLUDE_EMPTY_GOLD_RECORDS = True


# Feature ablation switches. Any feature omitted here is enabled.
# Disabled features are removed consistently from extraction, threshold
# calibration, gold targets, hierarchy expansion, scoring, and bootstrapping.
FEATURE_SWITCHES: Dict[str, bool] = {
    "State": False,
    "State_of_Matter": False,
    "Shape": False,
    "Rigidity": False,
}

# Sentence-cluster bootstrap, matching the RoBERTa evaluation design.
TEST_BOOTSTRAP_SAMPLES = 1700
TEST_BOOTSTRAP_CONFIDENCE = 0.95
TEST_BOOTSTRAP_SEED = 20260926


# =============================================================================
# GLiNER2.5 extraction settings
# =============================================================================

# Keep this low. Scores below this value are censored by the extractor and
# therefore cannot later be recovered by threshold optimisation.
MODEL_OUTPUT_THRESHOLD = 0.001

LONG_CHUNK_SIZE = 48
LONG_CHUNK_OVERLAP = 8

# Multiple semantic categories can legitimately cover the same noun span.
OVERLAP_POLICY = "allow"

# Used only when threshold optimisation is impossible for a feature, e.g. no
# positive validation examples.
DEFAULT_FINAL_THRESHOLD = 0.50

# Reuse expensive raw GLiNER extraction if the cache contains exactly the same
# feature-score inventory. Set False to force re-extraction.
REUSE_COMPATIBLE_RAW_SCORE_CACHE = True

SAVE_MATCHED_ENTITIES = True
SAVE_UNMATCHED_SPANS = True


# =============================================================================
# Feature hierarchy
# =============================================================================

# child -> direct parent(s)
FEATURE_PARENTS: Dict[str, Set[str]] = {

    "Organisation": {
        "Internal_Control",
        "Abstract",
    },

    "Natural_Force": {
        "Internal_Control",
    },

    "Animate": {
        "Internal_Control",
        "Natural",
    },

    "Human": {
        "Animate",
    },

    "Feminine": {
        "Human",
    },

    "Masculine": {
        "Human",
    },

    "Unspecified_Gender_or_Neuter": {
        "Human",
    },

    "Animal": {
        "Animate",
    },

    "Plant": {
        "Animate",
    },

    "Body_Part": {
        "Animate",
    },

    "Machine": {
        "Internal_Control",
        "Artefact",
    },

    "Vehicle": {
        "Machine",
    },

    "Natural": {
        "Concrete",
    },

    "Artefact": {
        "Concrete",
    },

    "Comestible": {
        "Concrete",
    },

    "Substance": {
        "Concrete",
    },

    "State_of_Matter": {
        "Concrete",
    },

    "Inanimate": {
        "Concrete",
    },

    "Garment": {
        "Artefact",
    },

    "Solid": {
        "State_of_Matter",
    },

    "Fluid": {
        "State_of_Matter",
    },

    "Rigidity": {
        "Solid",
    },

    "Shape": {
        "Solid",
    },

    "Rigid": {
        "Rigidity",
    },

    "Non_Rigid": {
        "Rigidity",
    },

    "Elongated": {
        "Shape",
    },

    "Pointy": {
        "Shape",
    },

    "Point_in_Time": {
        "Time",
    },

    "Duration": {
        "Time",
    },

    "Time": {
        "Abstract",
    },

    "Language": {
        "Abstract",
    },

    "Sound": {
        "Abstract",
    },

    "Communication": {
        "Abstract",
    },

    "Eventive": {
        "Abstract",
    },

    "Statement": {
        "Communication",
    },

    "Question": {
        "Communication",
    },
}


# =============================================================================
# GLiNER entity descriptions
# =============================================================================

# IMPORTANT:
# These are DIRECT extraction categories, not merely structural leaves.
# Internal ontology nodes are included because they may apply without any
# child applying. For example, a power saw is Machine but not Vehicle.
#
# Web_Location is intentionally capitalised exactly this way everywhere.

FEATURE_DESCRIPTIONS: Dict[str, str] = {

    "Abstract":
        "An abstract entity, concept, quality, condition, "
        "relation, informational object, event-like abstraction, or other "
        "non-physical referent.",

    "Animal":
        "A non-human animal, individual animal, animal kind, "
        "or animal species such as dog, cat, wolf, rat, bird, chicken, rabbit, "
        "snake, pigeon, horse, or elephant.",

    "Animate":
        "A living animate entity capable of biological agency, "
        "including humans, non-human animals, and living organisms treated as "
        "animate.",

    "Artefact":
        "A human-made physical object or product, such as a "
        "tool, device, piece of furniture, building, container, machine, "
        "garment, or manufactured object.",

    "Body_Part":
        "A body part, anatomical structure, organ, tissue, "
        "limb, or anatomical component such as eye, hair, tooth, face, nose, "
        "ear, finger, arm, leg, heart, or lung.",

    "Comestible":
        "Food or drink that can be consumed, such as food, water, milk, tea, juice, cheese, bread, "
        "rice, beef, breakfast, lunch, or dinner.",

    "Communication":
        "A communicative or informational object or act, such "
        "as a message, communication, utterance, conversation, report, "
        "statement, question, announcement, or request.",

    "Concrete":
        "A concrete entity with physical or spatial existence, "
        "including living organisms, objects, materials, substances, and other "
        "physically instantiated referents.",

    "Currency":
        "Money, currency, cash, coins, banknotes, or a named "
        "monetary unit such as dollars, euros, pounds, yen, or rupees.",

    "Duration":
        "An amount, interval, or duration of elapsed time, such "
        "as five days, a month, a period, several years, minutes, seconds, or "
        "an hour.",

    "Elongated":
        "A concrete entity whose salient physical shape is "
        "substantially longer in one dimension than the others, such as a rod, "
        "pencil, pen, finger, knife, stick, branch, or pole.",

    "Eventive":
        "An event, action, process, occurrence, happening, "
        "activity, ceremony, change, meeting, attack, arrival, development, "
        "or other temporally situated episode.",

    "Feminine":
        "A human whose expression identifies a feminine "
        "referent, such as woman, girl, daughter, queen, princess, actress, "
        "waitress, or a clearly feminine personal name in context.",

    "Fluid":
        "A non-solid matter capable of flowing, including liquids "
        "and gases such as water, milk, juice, air, oxygen, oil, gas, tea, or "
        "coffee.",

    "Garment":
        "Clothing, apparel, footwear, or another artefact "
        "principally worn on the body, such as trousers, shirt, hat, "
        "sunglasses, sweater, socks, gloves, scarf, or shoes.",

    "Human":
        "A human person or human being, including named people, "
        "roles, occupations, kinship terms, and generic person nouns.",

    "Inanimate":
        "A concrete non-living entity lacking biological "
        "animacy, such as a rock, mountain, table, chair, car, wall, house, "
        "road, building, bag, lamp, or tool.",

    "Internal_Control":
        "An entity capable of internally controlled or "
        "self-directed causal behaviour, including animate beings, "
        "organisations, machines, and internally acting causal systems.",

    "Language":
        "A language, dialect, linguistic variety, code, or "
        "linguistic system such as English, French, Thai, Japanese, Spanish, "
        "or Maori.",

    "Location":
        "A spatial location, place, site, town, city, region, "
        "country, geographic area, or other physical place.",

    "Machine":
        "A machine or powered/mechanical device such as a car, "
        "computer, lawnmower, phone, power saw, drill, printer, engine, or "
        "other machine.",

    "Masculine":
        "A human whose expression identifies a masculine "
        "referent, such as man, boy, son, king, prince, actor, waiter, or a "
        "clearly masculine personal name in context.",

    "Natural":
        "A naturally occurring concrete entity rather than a "
        "human-made artefact, including organisms, natural objects, landforms, "
        "and naturally occurring physical entities.",

    "Natural_Force":
        "A naturally occurring force or phenomenon capable "
        "of acting causally, such as wind, gravity, lightning, energy, a storm, "
        "a current, the sun, or another natural force.",

    "Non_Rigid":
        "A concrete object or material whose physical form is "
        "flexible, deformable, soft, bendable, or otherwise non-rigid, such as "
        "fabric, clothing, cotton, paper, or a soft bag.",

    "Organisation":
        "An organisation, social group, polity, government, "
        "institution, company, agency, association, team, committee, department, "
        "or other organised social entity. Include locations when used "
        "metonymically to refer to an organisation or government.",

    "Plant":
        "A plant organism or plant kind, including a tree, "
        "sapling, bush, shrub, flower, crop, fungus, grass, or similar botanical "
        "organism.",

    "Point_in_Time":
        "A particular point or location in time, such as an "
        "instant, date, clock time, particular day, named day, or specific "
        "moment.",

    "Pointy":
        "A concrete entity whose salient physical shape "
        "contains a tapering point, sharp tip, spike, or sharply narrowing end.",

    "Question":
        "A communicative object whose content asks for "
        "information, such as a question, inquiry, enquiry, query, or request "
        "for information.",

    "Rigid":
        "A concrete object or material whose physical form "
        "resists bending, compression, or deformation under ordinary conditions.",

    # "Rigidity":
    #     "A solid physical entity for which rigidity, stiffness, "
    #     "flexibility, or resistance to deformation is a relevant physical "
    #     "property.",

    # "Shape":
    #     "A span denoting a solid concrete entity characterised by a salient "
    #     "physical shape or geometric form.",

    "Solid":
        "A solid object or a concrete entity whose material state "
        "is solid rather than liquid or gaseous.",

    "Sound":
        "A sound, noise, tone, auditory signal, voice, ringing, "
        "bang, or another entity construed primarily as auditory.",

    # "State":
    #     "A condition, status, situation, or state in which an "
    #     "entity exists, excluding geographic or political states when they "
    #     "refer to places or governments.",

    # "State_of_Matter":
    #     "Matter or a physical entity viewed in terms of its "
    #     "physical state, especially solid, liquid, or gas.",

    "Statement":
        "A communicative object whose content makes an "
        "assertion, claim, announcement, declaration, report, proposition, "
        "statement, answer, allegation, or assertion.",

    "Substance":
        "A material or substance construed as matter rather "
        "than as an individuated object, such as water, gold, steel, sand, oil, "
        "milk, flour, wood, plastic, or air.",

    "Time":
        "A time or a temporal entity, including temporal periods, "
        "moments, dates, durations, eras, intervals, or temporal concepts.",

    "Unspecified_Gender_or_Neuter":
        "A human person or human role whose expression "
        "does not specify feminine or masculine gender, such as doctor, pilot, "
        "patient, professor, employee, expert, parent, worker, or person.",

    "Vehicle":
        "A vehicle or conveyance used to transport people or "
        "things, such as a car, ship, aircraft, train, van, truck, bus, "
        "motorcycle, or bicycle.",

    "Web_Location":
        "A website, webpage, web address, URL, internet domain, "
        "or named online location.",
}


# Every described category is directly extractable unless disabled above.
def feature_is_enabled(feature: str) -> bool:
    """Return whether a semantic feature participates in this experiment."""
    return bool(FEATURE_SWITCHES.get(feature, True))


ACTIVE_FEATURE_DESCRIPTIONS: Dict[str, str] = {
    feature: description
    for feature, description in FEATURE_DESCRIPTIONS.items()
    if feature_is_enabled(feature)
}

EXTRACTION_FEATURES = sorted(ACTIVE_FEATURE_DESCRIPTIONS)


# =============================================================================
# Mutually exclusive direct categories
# =============================================================================

EXCLUSIVE_GROUPS: List[Set[str]] = [
    {
        "Feminine",
        "Masculine",
        "Unspecified_Gender_or_Neuter",
    },
    {
        "Rigid",
        "Non_Rigid",
    },
]


# =============================================================================
# Hierarchy utilities
# =============================================================================

def feature_closure(features: Iterable[str]) -> Set[str]:
    """
    Return enabled features plus every enabled deterministic ancestor.

    Disabled hierarchy nodes are skipped in the returned set but traversal
    continues through them, so disabling an intermediate parent does not sever
    inheritance to higher enabled ancestors.
    """

    result: Set[str] = set()
    visited: Set[str] = set()
    stack = list(features)

    while stack:
        feature = stack.pop()

        if feature in visited:
            continue

        visited.add(feature)

        if feature_is_enabled(feature):
            result.add(feature)

        for parent in FEATURE_PARENTS.get(feature, set()):
            if parent not in visited:
                stack.append(parent)

    return result


def reduce_to_independent_features(features: Iterable[str]) -> Set[str]:
    """Remove features inherited from another feature already present."""

    feature_set = set(features)
    inherited: Set[str] = set()

    for feature in feature_set:
        inherited.update(feature_closure({feature}) - {feature})

    return feature_set - inherited


def resolve_exclusive_groups(
    selected: Set[str],
    scores: Dict[str, float],
) -> Set[str]:
    """Keep only the highest-scoring member of each exclusive group."""

    selected = set(selected)

    for group in EXCLUSIVE_GROUPS:
        active = selected & group

        if len(active) <= 1:
            continue

        winner = max(
            active,
            key=lambda label: scores.get(label, 0.0),
        )

        selected.difference_update(active)
        selected.add(winner)

    return selected


def select_direct_features(
    scores: Dict[str, float],
    thresholds: Dict[str, float],
) -> Set[str]:
    """Apply frozen per-feature thresholds to direct GLiNER scores."""

    selected: Set[str] = set()

    for feature in EXTRACTION_FEATURES:
        threshold = float(
            thresholds.get(feature, DEFAULT_FINAL_THRESHOLD)
        )

        if float(scores.get(feature, 0.0)) >= threshold:
            selected.add(feature)

    return resolve_exclusive_groups(selected, scores)


# =============================================================================
# CoNLL-U reading
# =============================================================================

def reconstruct_sentence(tokens: List[Dict[str, Any]]) -> str:
    parts: List[str] = []

    for token in tokens:
        parts.append(token["form"])

        if "SpaceAfter=No" not in token["misc"]:
            parts.append(" ")

    return "".join(parts).strip()


def iter_conllu_sentences(path: Path) -> Iterable[Dict[str, Any]]:
    """Read CoNLL-U while preserving document and paragraph IDs."""

    metadata: Dict[str, str] = {}
    tokens: List[Dict[str, Any]] = []

    current_doc_id: Optional[str] = None
    current_par_id: Optional[str] = None
    sentence_index = 0

    def emit_sentence() -> Optional[Dict[str, Any]]:
        nonlocal sentence_index

        if not tokens:
            return None

        text = metadata.get("text")

        if not text:
            text = reconstruct_sentence(tokens)

        output = {
            "sentence_index": sentence_index,
            "sent_id": metadata.get("sent_id"),
            "doc_id": current_doc_id,
            "par_id": current_par_id,
            "text": text,
            "tokens": list(tokens),
        }

        sentence_index += 1
        return output

    with path.open("r", encoding="utf-8") as infile:
        for raw_line in infile:
            line = raw_line.rstrip("\n")

            if not line:
                sentence = emit_sentence()

                if sentence is not None:
                    yield sentence

                metadata = {}
                tokens = []
                continue

            if line.startswith("#"):
                if "=" in line:
                    key, value = line[1:].split("=", 1)
                    key = key.strip()
                    value = value.strip()
                    metadata[key] = value

                    if key == "newdoc id":
                        current_doc_id = value
                        current_par_id = None
                    elif key == "newpar id":
                        current_par_id = value

                continue

            fields = line.split("\t")

            if len(fields) != 10:
                continue

            token_id = fields[0]

            # Ignore multiword-token rows and empty nodes.
            if "-" in token_id or "." in token_id:
                continue

            try:
                numeric_id = int(token_id)
            except ValueError:
                continue

            misc = fields[9]

            if misc == "_":
                misc = ""

            tokens.append(
                {
                    "id": numeric_id,
                    "form": fields[1],
                    "lemma": fields[2],
                    "upos": fields[3],
                    "xpos": fields[4],
                    "feats": fields[5],
                    "head": fields[6],
                    "deprel": fields[7],
                    "deps": fields[8],
                    "misc": misc,
                }
            )

    sentence = emit_sentence()

    if sentence is not None:
        yield sentence


# =============================================================================
# Character alignment and document construction
# =============================================================================

def align_tokens_to_text(
    text: str,
    tokens: List[Dict[str, Any]],
) -> List[Tuple[int, int]]:
    """Sequentially align CoNLL-U token forms to sentence text."""

    spans: List[Tuple[int, int]] = []
    cursor = 0

    for token in tokens:
        form = str(token["form"])
        start = text.find(form, cursor)

        if start < 0:
            raise ValueError(
                f"Could not align token {form!r} after character {cursor} "
                f"in sentence:\n{text}"
            )

        end = start + len(form)
        spans.append((start, end))
        cursor = end

    return spans


def prepare_sentence_alignment(
    sentence: Dict[str, Any],
) -> Tuple[str, List[Tuple[int, int]], bool]:
    """Prefer original text; fall back to token reconstruction if necessary."""

    original_text = str(sentence["text"])
    tokens = sentence["tokens"]

    try:
        spans = align_tokens_to_text(original_text, tokens)
        return original_text, spans, False
    except ValueError:
        fallback_text = reconstruct_sentence(tokens)
        spans = align_tokens_to_text(fallback_text, tokens)
        return fallback_text, spans, True


def split_into_documents(
    sentences: List[Dict[str, Any]],
) -> List[List[Dict[str, Any]]]:
    """Group contiguous sentences by document ID."""

    documents: List[List[Dict[str, Any]]] = []
    current: List[Dict[str, Any]] = []
    current_doc_id: Optional[str] = None

    for sentence in sentences:
        doc_id = sentence.get("doc_id")

        if not current:
            current = [sentence]
            current_doc_id = doc_id
            continue

        if doc_id == current_doc_id:
            current.append(sentence)
        else:
            documents.append(current)
            current = [sentence]
            current_doc_id = doc_id

    if current:
        documents.append(current)

    return documents


def make_token_key(
    sentence_index: int,
    token_id: int,
) -> Tuple[int, int]:
    return int(sentence_index), int(token_id)


def build_document(
    document_sentences: List[Dict[str, Any]],
    document_number: int,
) -> Dict[str, Any]:
    """Build document text and global NOUN/PROPN token offsets."""

    text_parts: List[str] = []
    target_tokens: List[Dict[str, Any]] = []
    document_cursor = 0
    fallback_sentence_count = 0

    for sentence_number, sentence in enumerate(document_sentences):
        sentence_text, token_spans, used_fallback = prepare_sentence_alignment(
            sentence
        )

        if used_fallback:
            fallback_sentence_count += 1

        sentence_start = document_cursor

        for token, local_span in zip(sentence["tokens"], token_spans):
            if token["upos"] not in TARGET_POS:
                continue

            local_start, local_end = local_span
            global_start = sentence_start + local_start
            global_end = sentence_start + local_end

            target_tokens.append(
                {
                    "key": make_token_key(
                        sentence["sentence_index"],
                        token["id"],
                    ),
                    "sentence_index": sentence["sentence_index"],
                    "doc_id": sentence.get("doc_id"),
                    "par_id": sentence.get("par_id"),
                    "sent_id": sentence.get("sent_id"),
                    "sentence_number": sentence_number,
                    "token_id": token["id"],
                    "form": token["form"],
                    "lemma": token["lemma"],
                    "upos": token["upos"],
                    "sentence": sentence["text"],
                    "model_sentence": sentence_text,
                    "start": global_start,
                    "end": global_end,
                }
            )

        text_parts.append(sentence_text)
        document_cursor += len(sentence_text)

        if sentence_number < len(document_sentences) - 1:
            text_parts.append("\n")
            document_cursor += 1

    document_text = "".join(text_parts)

    document_id = (
        document_sentences[0].get("doc_id")
        if document_sentences
        else None
    )

    if document_id is None:
        document_id = f"document_{document_number}"

    return {
        "document_id": document_id,
        "text": document_text,
        "target_tokens": target_tokens,
        "fallback_sentence_count": fallback_sentence_count,
    }


# =============================================================================
# GLiNER result parsing
# =============================================================================

def deduplicate_predictions(
    predictions: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Keep the highest-confidence copy of duplicate long-chunk predictions."""

    best: Dict[Tuple[str, int, int], Dict[str, Any]] = {}

    for prediction in predictions:
        key = (
            prediction["label"],
            prediction["start"],
            prediction["end"],
        )

        previous = best.get(key)

        if (
            previous is None
            or prediction["confidence"] > previous["confidence"]
        ):
            best[key] = prediction

    return sorted(
        best.values(),
        key=lambda item: (
            item["start"],
            item["end"],
            item["label"],
        ),
    )


def parse_entity_predictions(
    result: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """Flatten GLiNER2's entity output to labelled span records."""

    predictions: List[Dict[str, Any]] = []
    entities = result.get("entities", {})

    if not isinstance(entities, dict):
        return predictions

    for label, values in entities.items():
        label = str(label)

        if label not in EXTRACTION_FEATURES:
            continue

        if values is None:
            continue

        if not isinstance(values, list):
            values = [values]

        for item in values:
            if not isinstance(item, dict):
                continue

            start = item.get("start")
            end = item.get("end")

            if start is None or end is None:
                continue

            confidence = item.get(
                "confidence",
                item.get("score", 0.0),
            )

            try:
                confidence = float(confidence)
                start = int(start)
                end = int(end)
            except (TypeError, ValueError):
                continue

            if end <= start:
                continue

            predictions.append(
                {
                    "label": label,
                    "text": str(item.get("text", "")),
                    "confidence": confidence,
                    "start": start,
                    "end": end,
                }
            )

    return deduplicate_predictions(predictions)


# =============================================================================
# Match GLiNER spans back to UD NOUN/PROPN tokens
# =============================================================================

def interval_overlap(
    start_a: int,
    end_a: int,
    start_b: int,
    end_b: int,
) -> int:
    return max(0, min(end_a, end_b) - max(start_a, start_b))


def match_prediction_to_tokens(
    prediction: Dict[str, Any],
    target_tokens: List[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], str]:
    """Map one predicted semantic span onto overlapping target nouns."""

    entity_start = prediction["start"]
    entity_end = prediction["end"]

    contained: List[Dict[str, Any]] = []

    for token in target_tokens:
        if token["start"] >= entity_start and token["end"] <= entity_end:
            contained.append(token)

    if contained:
        return contained, "contained"

    overlapping: List[Dict[str, Any]] = []

    for token in target_tokens:
        overlap = interval_overlap(
            entity_start,
            entity_end,
            token["start"],
            token["end"],
        )

        if overlap <= 0:
            continue

        token_length = max(1, token["end"] - token["start"])
        entity_length = max(1, entity_end - entity_start)
        denominator = min(token_length, entity_length)
        overlap_ratio = overlap / denominator

        if overlap_ratio >= 0.50:
            overlapping.append(token)

    if overlapping:
        return overlapping, "overlap"

    return [], "none"


def assign_predictions_to_tokens(
    target_tokens: List[Dict[str, Any]],
    predictions: List[Dict[str, Any]],
) -> Tuple[
    Dict[Tuple[int, int], Dict[str, float]],
    Dict[Tuple[int, int], List[Dict[str, Any]]],
    List[Dict[str, Any]],
]:
    """Convert extracted spans to per-token maximum feature confidence scores."""

    token_scores: Dict[Tuple[int, int], Dict[str, float]] = {}
    token_matches: Dict[Tuple[int, int], List[Dict[str, Any]]] = {}

    for token in target_tokens:
        token_scores[token["key"]] = {
            feature: 0.0
            for feature in EXTRACTION_FEATURES
        }
        token_matches[token["key"]] = []

    unmatched: List[Dict[str, Any]] = []

    for prediction in predictions:
        matched_tokens, match_type = match_prediction_to_tokens(
            prediction,
            target_tokens,
        )

        if not matched_tokens:
            unmatched.append(prediction)
            continue

        for token in matched_tokens:
            key = token["key"]
            label = prediction["label"]
            confidence = prediction["confidence"]

            token_scores[key][label] = max(
                token_scores[key][label],
                confidence,
            )

            token_matches[key].append(
                {
                    "label": label,
                    "text": prediction["text"],
                    "confidence": confidence,
                    "start": prediction["start"],
                    "end": prediction["end"],
                    "match_type": match_type,
                }
            )

    return token_scores, token_matches, unmatched


# =============================================================================
# Raw-score cache
# =============================================================================

def current_raw_cache_config() -> Dict[str, Any]:
    """
    Return the extraction settings that materially determine cached raw scores.

    Old caches created with a different raw proposal floor (for example 0.01)
    must NOT be reused after lowering MODEL_OUTPUT_THRESHOLD to 0.001.
    """

    description_payload = json.dumps(
        {
            feature: ACTIVE_FEATURE_DESCRIPTIONS[feature]
            for feature in EXTRACTION_FEATURES
        },
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return {
        "model": MODEL_NAME,
        "raw_extraction_threshold": float(MODEL_OUTPUT_THRESHOLD),
        "chunk_size": int(LONG_CHUNK_SIZE),
        "chunk_overlap": int(LONG_CHUNK_OVERLAP),
        "overlap_policy": OVERLAP_POLICY,
        "feature_switches": dict(FEATURE_SWITCHES),
        "extraction_features": list(EXTRACTION_FEATURES),
        "feature_description_sha256": hashlib.sha256(
            description_payload.encode("utf-8")
        ).hexdigest(),
    }


def raw_score_cache_is_compatible(path: Path) -> bool:
    """
    Check that an existing raw cache was created with the CURRENT extraction
    configuration, not merely the same feature names.
    """

    if not path.exists():
        return False

    expected_features = set(EXTRACTION_FEATURES)
    expected_config = current_raw_cache_config()
    checked = 0

    try:
        with path.open("r", encoding="utf-8") as handle:
            for line in handle:
                line = line.strip()

                if not line:
                    continue

                record = json.loads(line)
                scores = record.get("feature_scores")
                cache_config = record.get("_cache_config")

                if not isinstance(scores, dict):
                    return False

                if set(scores) != expected_features:
                    return False

                # Caches produced by older versions have no configuration
                # metadata and are deliberately treated as incompatible.
                if cache_config != expected_config:
                    return False

                checked += 1

                if checked >= 20:
                    break
    except Exception:
        return False

    return checked > 0

def extract_raw_scores_for_corpus(
    model: Any,
    conllu_path: Path,
    output_path: Path,
    unmatched_path: Path,
    corpus_name: str,
) -> None:
    """Run native GLiNER span extraction and save unthresholded token scores."""

    if (
        REUSE_COMPATIBLE_RAW_SCORE_CACHE
        and raw_score_cache_is_compatible(output_path)
    ):
        print()
        print(
            f"Reusing compatible {corpus_name} raw-score cache:\n"
            f"  {output_path.resolve()}"
        )
        return

    if not conllu_path.exists():
        raise FileNotFoundError(
            f"CoNLL-U file not found:\n{conllu_path.resolve()}"
        )

    print()
    print("=" * 100)
    print(f"RAW GLiNER EXTRACTION — {corpus_name.upper()}")
    print("=" * 100)

    sentences = list(iter_conllu_sentences(conllu_path))
    documents = split_into_documents(sentences)

    noun_count = sum(
        1
        for sentence in sentences
        for token in sentence["tokens"]
        if token["upos"] in TARGET_POS
    )

    print(f"Sentences       : {len(sentences):,}")
    print(f"Documents       : {len(documents):,}")
    print(f"NOUN/PROPN      : {noun_count:,}")
    print(f"Feature classes : {len(EXTRACTION_FEATURES):,}")

    total_spans = 0
    total_unmatched = 0
    fallback_sentences = 0

    unmatched_handle = (
        unmatched_path.open("w", encoding="utf-8")
        if SAVE_UNMATCHED_SPANS
        else None
    )

    try:
        with output_path.open("w", encoding="utf-8") as outfile:
            for document_number, document_sentences in enumerate(
                tqdm(documents, desc=f"{corpus_name} documents")
            ):
                document = build_document(
                    document_sentences,
                    document_number,
                )

                document_text = document["text"]
                target_tokens = document["target_tokens"]
                fallback_sentences += document["fallback_sentence_count"]

                if not document_text.strip():
                    continue

                result = model.extract_entities_long(
                    document_text,
                    ACTIVE_FEATURE_DESCRIPTIONS,
                    chunk_size=LONG_CHUNK_SIZE,
                    chunk_overlap=LONG_CHUNK_OVERLAP,
                    threshold=MODEL_OUTPUT_THRESHOLD,
                    include_confidence=True,
                    include_spans=True,
                    overlap_policy=OVERLAP_POLICY,
                )

                predictions = parse_entity_predictions(result)
                total_spans += len(predictions)

                (
                    token_scores,
                    token_matches,
                    unmatched_predictions,
                ) = assign_predictions_to_tokens(
                    target_tokens,
                    predictions,
                )

                total_unmatched += len(unmatched_predictions)

                if unmatched_handle is not None:
                    for prediction in unmatched_predictions:
                        unmatched_handle.write(
                            json.dumps(
                                {
                                    "doc_id": document["document_id"],
                                    **prediction,
                                },
                                ensure_ascii=False,
                            )
                            + "\n"
                        )

                for token in target_tokens:
                    key = token["key"]

                    record: Dict[str, Any] = {
                        "_cache_config": current_raw_cache_config(),
                        "sentence_index": token["sentence_index"],
                        "doc_id": token["doc_id"],
                        "par_id": token["par_id"],
                        "sent_id": token["sent_id"],
                        "token_id": token["token_id"],
                        "form": token["form"],
                        "lemma": token["lemma"],
                        "upos": token["upos"],
                        "sentence": token["sentence"],
                        "feature_scores": {
                            feature: float(token_scores[key].get(feature, 0.0))
                            for feature in EXTRACTION_FEATURES
                        },
                    }

                    if SAVE_MATCHED_ENTITIES:
                        record["matched_entities"] = sorted(
                            token_matches[key],
                            key=lambda item: (
                                -item["confidence"],
                                item["label"],
                            ),
                        )

                    outfile.write(
                        json.dumps(record, ensure_ascii=False)
                        + "\n"
                    )

            outfile.flush()

        if unmatched_handle is not None:
            unmatched_handle.flush()

    finally:
        if unmatched_handle is not None:
            unmatched_handle.close()

    print()
    print(f"Raw extracted spans              : {total_spans:,}")
    print(f"Raw unmatched spans              : {total_unmatched:,}")
    print(f"Sentence alignment fallbacks     : {fallback_sentences:,}")
    print(f"Raw token-score output           : {output_path.resolve()}")


# =============================================================================
# JSONL loading
# =============================================================================

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"File not found:\n{path.resolve()}")

    records: List[Dict[str, Any]] = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON on line {line_number} of {path}: {exc}"
                ) from exc

            if isinstance(record, dict):
                records.append(record)

    return records


def clean_feature_list(value: Any) -> Set[str]:
    """Return enabled, non-empty semantic feature names."""
    if not isinstance(value, list):
        return set()

    return {
        str(feature).strip()
        for feature in value
        if str(feature).strip()
        and feature_is_enabled(str(feature).strip())
    }


def load_gold_records(
    path: Path,
) -> Dict[Tuple[int, int], Dict[str, Any]]:
    """Load gold NOUN/PROPN annotations keyed by sentence/token ID."""

    records: Dict[Tuple[int, int], Dict[str, Any]] = {}

    for record in load_jsonl(path):
        if record.get("upos") not in TARGET_POS:
            continue

        sentence_index = record.get("sentence_index")
        token_id = record.get("token_id")

        if not isinstance(sentence_index, int):
            continue

        if not isinstance(token_id, int):
            continue

        key = make_token_key(sentence_index, token_id)
        records[key] = record

    if not records:
        raise RuntimeError(
            "No usable gold training NOUN/PROPN records were loaded."
        )

    return records


def load_raw_score_records(
    path: Path,
) -> Dict[Tuple[int, int], Dict[str, Any]]:
    records: Dict[Tuple[int, int], Dict[str, Any]] = {}

    for record in load_jsonl(path):
        sentence_index = record.get("sentence_index")
        token_id = record.get("token_id")

        if not isinstance(sentence_index, int):
            continue

        if not isinstance(token_id, int):
            continue

        key = make_token_key(sentence_index, token_id)
        records[key] = record

    return records


# =============================================================================
# Validation split
# =============================================================================

def _build_sentence_split_inventory(
    gold_records: Dict[Tuple[int, int], Dict[str, Any]],
) -> Tuple[
    List[int],
    np.ndarray,
    List[str],
    Dict[int, Counter],
]:
    """
    Build the same sentence-level multilabel inventory used by RoBERTa.

    A sentence is eligible iff it contains at least one NOUN/PROPN record with
    at least one enabled semantic feature. Stratification uses presence of the
    per-token independent targets, while a separate token-count inventory is
    retained for the minimum-training-support constraint.
    """

    sentence_feature_counts: Dict[int, Counter] = defaultdict(Counter)

    for (sentence_index, _), record in gold_records.items():
        features = clean_feature_list(record.get("features", []))

        if EXCLUDE_EMPTY_GOLD_RECORDS and not features:
            continue

        if not features:
            continue

        independent = reduce_to_independent_features(
            feature_closure(features)
        )

        for feature in independent:
            sentence_feature_counts[int(sentence_index)][feature] += 1

    eligible = sorted(
        sentence_index
        for sentence_index, counts in sentence_feature_counts.items()
        if counts
    )

    if len(eligible) < 2:
        raise RuntimeError(
            "Need at least two eligible training sentences for validation split."
        )

    stratification_labels = sorted(
        {
            feature
            for sentence_index in eligible
            for feature, count in sentence_feature_counts[sentence_index].items()
            if count > 0
        }
    )

    if not stratification_labels:
        raise RuntimeError(
            "No enabled independent features were found in eligible sentences."
        )

    label2column = {
        label: column
        for column, label in enumerate(stratification_labels)
    }

    sentence_matrix = np.zeros(
        (len(eligible), len(stratification_labels)),
        dtype=np.int8,
    )

    for row, sentence_index in enumerate(eligible):
        for feature in sentence_feature_counts[sentence_index]:
            sentence_matrix[row, label2column[feature]] = 1

    return (
        eligible,
        sentence_matrix,
        stratification_labels,
        dict(sentence_feature_counts),
    )


def _validate_candidate_split(
    train_indices: Sequence[int],
    validation_indices: Sequence[int],
    eligible: Sequence[int],
) -> Tuple[bool, str]:
    """Validate exact coverage and the expected validation sentence count."""

    train_set = set(int(x) for x in train_indices)
    validation_set = set(int(x) for x in validation_indices)
    eligible_set = set(int(x) for x in eligible)

    if train_set & validation_set:
        return False, "train/validation sentence overlap"

    if train_set | validation_set != eligible_set:
        missing = len(eligible_set - (train_set | validation_set))
        extra = len((train_set | validation_set) - eligible_set)
        return (
            False,
            f"split does not exactly cover eligible sentences "
            f"(missing={missing}, extra={extra})",
        )

    # expected_validation = int(
    #     round(len(eligible) * VALIDATION_FRACTION)
    # )

    # if len(validation_set) != expected_validation:
    #     return (
    #         False,
    #         f"validation size is {len(validation_set):,}, expected "
    #         f"{expected_validation:,} for {VALIDATION_FRACTION:.0%}",
    #     )

    return True, "ok"


def _write_canonical_split(
    split: Dict[str, List[int]],
    source: str,
    stratification_labels: Sequence[str],
) -> None:
    """Persist the canonical split and a GLiNER provenance copy."""

    document = {
        "source": source,
        "method": "sentence_level_multilabel_stratification",
        "validation_fraction": float(VALIDATION_FRACTION),
        "split_seed": int(SPLIT_SEED),
        "minimum_train_positives_per_feature": int(
            MIN_TRAIN_POSITIVES_PER_FEATURE
        ),
        "feature_switches": dict(FEATURE_SWITCHES),
        "stratification_labels": list(stratification_labels),
        "train": list(split["train"]),
        "validation": list(split["validation"]),
    }

    SHARED_SPLIT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    GLINER_SPLIT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with SHARED_SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    gliner_document = dict(document)
    gliner_document["canonical_shared_split"] = str(
        SHARED_SPLIT_PATH
    )

    with GLINER_SPLIT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            gliner_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )


def _load_split_file(path: Path) -> Optional[Dict[str, List[int]]]:
    """Load train/validation arrays from a split JSON if structurally valid."""

    if not path.exists():
        return None

    try:
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)
    except Exception:
        return None

    train_indices = data.get("train")
    validation_indices = data.get("validation")

    if not isinstance(train_indices, list):
        return None

    if not isinstance(validation_indices, list):
        return None

    return {
        "train": sorted(int(x) for x in train_indices),
        "validation": sorted(int(x) for x in validation_indices),
    }


def _create_roberta_style_split(
    eligible: Sequence[int],
    sentence_matrix: np.ndarray,
    stratification_labels: Sequence[str],
    sentence_feature_counts: Dict[int, Counter],
) -> Dict[str, List[int]]:
    """
    Reconstruct the RoBERTa sentence-level split deterministically.

    Iterative multilabel stratification is retried only if a candidate removes
    every direct token occurrence of an enabled feature from the training set.
    """

    all_token_counts: Counter = Counter()

    for counts in sentence_feature_counts.values():
        all_token_counts.update(counts)

    x_dummy = np.zeros(
        (len(eligible), 1),
        dtype=np.int8,
    )

    best_candidate: Optional[Dict[str, Any]] = None
    rejected_for_coverage = 0

    for attempt in range(MAX_STRATIFICATION_ATTEMPTS):
        attempt_seed = SPLIT_SEED + attempt

        splitter = MultilabelStratifiedShuffleSplit(
            n_splits=1,
            test_size=VALIDATION_FRACTION,
            random_state=attempt_seed,
        )

        train_rows, validation_rows = next(
            splitter.split(x_dummy, sentence_matrix)
        )

        train_indices = [
            int(eligible[int(row)])
            for row in train_rows
        ]
        validation_indices = [
            int(eligible[int(row)])
            for row in validation_rows
        ]

        train_token_counts: Counter = Counter()

        for sentence_index in train_indices:
            train_token_counts.update(
                sentence_feature_counts[sentence_index]
            )

        undercovered = sorted(
            feature
            for feature, total_count in all_token_counts.items()
            if (
                total_count > 0
                and train_token_counts[feature]
                < MIN_TRAIN_POSITIVES_PER_FEATURE
            )
        )

        if undercovered:
            rejected_for_coverage += 1
            continue

        total_support = sentence_matrix.sum(
            axis=0
        ).astype(np.float64)
        validation_support = sentence_matrix[
            validation_rows
        ].sum(axis=0).astype(np.float64)

        target_support = (
            total_support * float(VALIDATION_FRACTION)
        )
        scale = np.maximum(target_support, 1.0)

        prevalence_error = float(
            np.mean(
                np.abs(
                    validation_support - target_support
                )
                / scale
            )
        )

        candidate = {
            "train": sorted(train_indices),
            "validation": sorted(validation_indices),
            "prevalence_error": prevalence_error,
            "split_seed_used": attempt_seed,
        }

        if (
            best_candidate is None
            or prevalence_error
            < best_candidate["prevalence_error"]
        ):
            best_candidate = candidate

        # Match the RoBERTa script: when the first candidate already preserves
        # coverage, use it immediately.
        if rejected_for_coverage == 0:
            break

    if best_candidate is None:
        raise RuntimeError(
            "Could not create a sentence-level multilabel-stratified split "
            f"after {MAX_STRATIFICATION_ATTEMPTS} deterministic attempts."
        )

    split = {
        "train": best_candidate["train"],
        "validation": best_candidate["validation"],
    }

    valid, reason = _validate_candidate_split(
        split["train"],
        split["validation"],
        eligible,
    )

    if not valid:
        raise RuntimeError(
            "Internally generated split failed validation: "
            f"{reason}."
        )

    print()
    print("Reconstructed canonical RoBERTa-style sentence split.")
    print(
        f"Eligible sentences   : {len(eligible):,}"
    )
    print(
        f"Training sentences   : {len(split['train']):,}"
    )
    print(
        f"Validation sentences : {len(split['validation']):,}"
    )
    print(
        "Split seed used      : "
        f"{best_candidate['split_seed_used']}"
    )
    print(
        "Coverage rejections  : "
        f"{rejected_for_coverage:,}"
    )
    print(
        "Prevalence error     : "
        f"{best_candidate['prevalence_error']:.6f}"
    )

    return split


def load_or_create_validation_split(
    gold_records: Dict[Tuple[int, int], Dict[str, Any]],
) -> Dict[str, List[int]]:
    """
    Load or construct one canonical model-independent sentence split.

    Priority:
      1. validated SHARED_SPLIT_PATH;
      2. validated legacy RoBERTa split;
      3. deterministic reconstruction using the RoBERTa split algorithm.

    A stale split with the wrong sentence count is NEVER accepted silently.
    """

    (
        eligible,
        sentence_matrix,
        stratification_labels,
        sentence_feature_counts,
    ) = _build_sentence_split_inventory(
        gold_records
    )

    expected_validation = int(
        round(len(eligible) * VALIDATION_FRACTION)
    )

    print()
    print("Sentence-level split inventory:")
    print(
        f"  Eligible sentences          : {len(eligible):,}"
    )
    print(
        f"  Expected validation (20%)  : {expected_validation:,}"
    )
    print(
        f"  Expected training           : "
        f"{len(eligible) - expected_validation:,}"
    )

    shared = _load_split_file(
        SHARED_SPLIT_PATH
    )

    if shared is not None:
        valid, reason = _validate_candidate_split(
            shared["train"],
            shared["validation"],
            eligible,
        )

        if valid:
            _write_canonical_split(
                shared,
                source=str(SHARED_SPLIT_PATH),
                stratification_labels=stratification_labels,
            )

            print()
            print("Using validated canonical shared sentence split.")
            print(
                f"Training sentences   : {len(shared['train']):,}"
            )
            print(
                f"Validation sentences : {len(shared['validation']):,}"
            )

            return shared

        print()
        print(
            "Ignoring incompatible canonical shared split:"
        )
        print(
            f"  {SHARED_SPLIT_PATH.resolve()}"
        )
        print(
            f"  Reason: {reason}"
        )

    legacy = _load_split_file(
        ROBERTA_SPLIT_PATH
    )

    if legacy is not None:
        valid, reason = _validate_candidate_split(
            legacy["train"],
            legacy["validation"],
            eligible,
        )

        if valid:
            _write_canonical_split(
                legacy,
                source=str(ROBERTA_SPLIT_PATH),
                stratification_labels=stratification_labels,
            )

            print()
            print(
                "Imported validated RoBERTa split into canonical shared split."
            )
            print(
                f"Training sentences   : {len(legacy['train']):,}"
            )
            print(
                f"Validation sentences : {len(legacy['validation']):,}"
            )

            return legacy

        print()
        print()
        print(
            "Legacy RoBERTa split exists but is incompatible and will "
            "NOT be reused."
        )
        print(
            f"  {ROBERTA_SPLIT_PATH.resolve()}"
        )
        print(
            f"  Train/validation sizes: "
            f"{len(legacy['train']):,}/"
            f"{len(legacy['validation']):,}"
        )
        print(
            f"  Reason: {reason}"
        )

    split = _create_roberta_style_split(
        eligible=eligible,
        sentence_matrix=sentence_matrix,
        stratification_labels=stratification_labels,
        sentence_feature_counts=sentence_feature_counts,
    )

    _write_canonical_split(
        split,
        source="reconstructed_by_gliner_with_roberta_method",
        stratification_labels=stratification_labels,
    )

    print()
    print(
        "Canonical shared split saved to:"
    )
    print(
        f"  {SHARED_SPLIT_PATH.resolve()}"
    )

    return split


# =============================================================================
# Threshold optimisation
# =============================================================================

def binary_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


def build_threshold_candidates(scores: np.ndarray) -> np.ndarray:
    """
    Use observed non-zero scores as exact decision boundaries.

    Zero means GLiNER did not return that feature at the raw extraction
    threshold, so thresholds below MODEL_OUTPUT_THRESHOLD are not meaningful.
    """

    usable = scores[
        np.isfinite(scores)
        & (scores >= MODEL_OUTPUT_THRESHOLD)
    ]

    candidates = list(np.unique(usable))
    candidates.append(float(MODEL_OUTPUT_THRESHOLD))
    candidates.append(float(DEFAULT_FINAL_THRESHOLD))
    candidates.append(1.0)

    return np.asarray(
        sorted(set(float(x) for x in candidates)),
        dtype=np.float64,
    )


def optimise_one_threshold(
    y_true: np.ndarray,
    scores: np.ndarray,
) -> Tuple[float, Dict[str, Any]]:
    """Choose the threshold maximising binary validation F1."""

    positive_support = int(y_true.sum())
    negative_support = int((1 - y_true).sum())

    default_pred = (
        scores >= DEFAULT_FINAL_THRESHOLD
    ).astype(np.int8)

    default_metrics = binary_metrics(
        y_true,
        default_pred,
    )

    if positive_support == 0 or negative_support == 0:
        return (
            DEFAULT_FINAL_THRESHOLD,
            {
                "threshold": DEFAULT_FINAL_THRESHOLD,
                "support": positive_support,
                "negative_support": negative_support,
                "status": "fallback_insufficient_class_variation",
                "precision_default": default_metrics["precision"],
                "recall_default": default_metrics["recall"],
                "f1_default": default_metrics["f1"],
                "precision_optimised": default_metrics["precision"],
                "recall_optimised": default_metrics["recall"],
                "f1_optimised": default_metrics["f1"],
                "f1_change": 0.0,
            },
        )

    candidates = build_threshold_candidates(scores)
    best_f1 = -1.0
    best_candidates: List[float] = []

    for threshold in candidates:
        y_pred = (
            scores >= threshold
        ).astype(np.int8)

        metrics = binary_metrics(
            y_true,
            y_pred,
        )

        f1 = metrics["f1"]

        if f1 > best_f1 + 1e-12:
            best_f1 = f1
            best_candidates = [float(threshold)]
        elif abs(f1 - best_f1) <= 1e-12:
            best_candidates.append(float(threshold))

    # Stable tie-breaking: prefer the threshold nearest 0.5, then the lower
    # threshold (slightly recall-favouring) if equally distant.
    best_threshold = min(
        best_candidates,
        key=lambda value: (
            abs(value - DEFAULT_FINAL_THRESHOLD),
            value,
        ),
    )

    best_pred = (
        scores >= best_threshold
    ).astype(np.int8)

    best_metrics = binary_metrics(
        y_true,
        best_pred,
    )

    return (
        float(best_threshold),
        {
            "threshold": float(best_threshold),
            "support": positive_support,
            "negative_support": negative_support,
            "status": "optimised",
            "precision_default": default_metrics["precision"],
            "recall_default": default_metrics["recall"],
            "f1_default": default_metrics["f1"],
            "precision_optimised": best_metrics["precision"],
            "recall_optimised": best_metrics["recall"],
            "f1_optimised": best_metrics["f1"],
            "f1_change": best_metrics["f1"] - default_metrics["f1"],
        },
    )


def optimise_thresholds_on_validation(
    gold_records: Dict[Tuple[int, int], Dict[str, Any]],
    raw_records: Dict[Tuple[int, int], Dict[str, Any]],
    validation_sentence_indices: Sequence[int],
) -> Tuple[
    Dict[str, float],
    Dict[str, Dict[str, Any]],
    List[Tuple[int, int]],
]:
    """
    Optimise one GLiNER threshold per directly extractable feature.

    Gold target for a feature is COMPLETE gold membership, not the reduced
    independent representation. This is intentional because GLiNER may validly
    predict a parent and child simultaneously; e.g. Machine remains true for a
    Vehicle instance.
    """

    validation_set = set(int(x) for x in validation_sentence_indices)
    keys: List[Tuple[int, int]] = []

    for key, gold_record in gold_records.items():
        sentence_index, _ = key

        if sentence_index not in validation_set:
            continue

        gold_features = clean_feature_list(
            gold_record.get("features", [])
        )

        if EXCLUDE_EMPTY_GOLD_RECORDS and not gold_features:
            continue

        if key not in raw_records:
            continue

        keys.append(key)

    keys.sort()

    if not keys:
        raise RuntimeError(
            "No aligned validation tokens were available for threshold tuning."
        )

    thresholds: Dict[str, float] = {}
    diagnostics: Dict[str, Dict[str, Any]] = {}

    for feature in EXTRACTION_FEATURES:
        y_true_values: List[int] = []
        score_values: List[float] = []

        for key in keys:
            gold_features = clean_feature_list(
                gold_records[key].get("features", [])
            )

            # Ensure gold is hierarchy-complete even if a source record omitted
            # an inherited parent.
            gold_complete = feature_closure(gold_features)

            scores = raw_records[key].get(
                "feature_scores",
                {},
            )

            try:
                score = float(scores.get(feature, 0.0))
            except (TypeError, ValueError):
                score = 0.0

            if not np.isfinite(score):
                score = 0.0

            y_true_values.append(
                1 if feature in gold_complete else 0
            )
            score_values.append(score)

        y_true = np.asarray(y_true_values, dtype=np.int8)
        score_array = np.asarray(score_values, dtype=np.float64)

        threshold, feature_diagnostics = optimise_one_threshold(
            y_true,
            score_array,
        )

        thresholds[feature] = threshold
        diagnostics[feature] = feature_diagnostics

    return thresholds, diagnostics, keys


# =============================================================================
# Validation metrics after all thresholds + exclusivity + inheritance
# =============================================================================

def calculate_set_micro_metrics(
    gold_sets: Sequence[Set[str]],
    pred_sets: Sequence[Set[str]],
) -> Dict[str, float]:
    tp = 0
    fp = 0
    fn = 0
    exact = 0

    for gold, pred in zip(gold_sets, pred_sets):
        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)

        if gold == pred:
            exact += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2.0 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    return {
        "precision_micro": float(precision),
        "recall_micro": float(recall),
        "f1_micro": float(f1),
        "exact_tagset_accuracy": (
            exact / len(gold_sets)
            if gold_sets
            else 0.0
        ),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
    }


def evaluate_validation_thresholds(
    keys: Sequence[Tuple[int, int]],
    gold_records: Dict[Tuple[int, int], Dict[str, Any]],
    raw_records: Dict[Tuple[int, int], Dict[str, Any]],
    thresholds: Dict[str, float],
) -> Dict[str, float]:
    gold_sets: List[Set[str]] = []
    pred_sets: List[Set[str]] = []

    for key in keys:
        gold_features = clean_feature_list(
            gold_records[key].get("features", [])
        )
        gold_complete = feature_closure(gold_features)

        scores = raw_records[key].get(
            "feature_scores",
            {},
        )

        direct = select_direct_features(
            scores,
            thresholds,
        )
        pred_complete = feature_closure(direct)

        gold_sets.append(gold_complete)
        pred_sets.append(pred_complete)

    return calculate_set_micro_metrics(
        gold_sets,
        pred_sets,
    )


def print_threshold_table(
    diagnostics: Dict[str, Dict[str, Any]],
) -> None:
    print()
    print("=" * 108)
    print("VALIDATION-OPTIMISED GLiNER THRESHOLDS")
    print("=" * 108)

    heading = (
        f"{'Feature':35s} "
        f"{'Threshold':>10s} "
        f"{'Support':>9s} "
        f"{'F1@0.5':>10s} "
        f"{'F1 opt':>10s} "
        f"{'Change':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in EXTRACTION_FEATURES:
        result = diagnostics[feature]

        print(
            f"{feature:35s} "
            f"{float(result['threshold']):10.4f} "
            f"{int(result['support']):9d} "
            f"{float(result['f1_default']):10.4f} "
            f"{float(result['f1_optimised']):10.4f} "
            f"{float(result['f1_change']):+10.4f}"
        )


# =============================================================================
# Comparable validation/test evaluation
# =============================================================================

def build_gold_label_vocabularies(
    gold_records: Dict[Tuple[int, int], Dict[str, Any]],
    training_sentence_indices: Sequence[int],
) -> Tuple[List[str], List[str]]:
    """
    Build full and independent evaluation vocabularies from TRAINING data only.
    """

    sentence_set = set(int(x) for x in training_sentence_indices)
    full_labels: Set[str] = set()
    independent_labels: Set[str] = set()

    for (sentence_index, _), record in gold_records.items():
        if sentence_index not in sentence_set:
            continue

        features = clean_feature_list(record.get("features", []))

        if EXCLUDE_EMPTY_GOLD_RECORDS and not features:
            continue

        complete = feature_closure(features)
        full_labels.update(complete)
        independent_labels.update(
            reduce_to_independent_features(complete)
        )

    if not full_labels:
        raise RuntimeError(
            "No enabled full semantic labels were found in the training split."
        )

    if not independent_labels:
        raise RuntimeError(
            "No enabled independent semantic labels were found in the training split."
        )

    return sorted(full_labels), sorted(independent_labels)


def _binary_matrix_from_sets(
    feature_sets: Sequence[Set[str]],
    labels: Sequence[str],
) -> np.ndarray:
    label2id = {
        label: index
        for index, label in enumerate(labels)
    }

    matrix = np.zeros(
        (len(feature_sets), len(labels)),
        dtype=np.int8,
    )

    for row, features in enumerate(feature_sets):
        for feature in features:
            label_id = label2id.get(feature)

            if label_id is not None:
                matrix[row, label_id] = 1

    return matrix


def _metrics_from_confusion_counts(
    tp_by_label: np.ndarray,
    fp_by_label: np.ndarray,
    fn_by_label: np.ndarray,
) -> Dict[str, float]:
    tp_by_label = tp_by_label.astype(np.float64)
    fp_by_label = fp_by_label.astype(np.float64)
    fn_by_label = fn_by_label.astype(np.float64)

    tp_total = float(tp_by_label.sum())
    fp_total = float(fp_by_label.sum())
    fn_total = float(fn_by_label.sum())

    precision_micro = (
        tp_total / (tp_total + fp_total)
        if (tp_total + fp_total) > 0
        else 0.0
    )

    recall_micro = (
        tp_total / (tp_total + fn_total)
        if (tp_total + fn_total) > 0
        else 0.0
    )

    f1_micro = (
        2.0 * precision_micro * recall_micro
        / (precision_micro + recall_micro)
        if (precision_micro + recall_micro) > 0
        else 0.0
    )

    precision_den = tp_by_label + fp_by_label
    recall_den = tp_by_label + fn_by_label

    precision_by_label = np.divide(
        tp_by_label,
        precision_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=precision_den > 0,
    )

    recall_by_label = np.divide(
        tp_by_label,
        recall_den,
        out=np.zeros_like(tp_by_label, dtype=np.float64),
        where=recall_den > 0,
    )

    f1_den = precision_by_label + recall_by_label

    f1_by_label = np.divide(
        2.0 * precision_by_label * recall_by_label,
        f1_den,
        out=np.zeros_like(precision_by_label, dtype=np.float64),
        where=f1_den > 0,
    )

    return {
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_micro": float(f1_micro),
        "precision_macro": float(precision_by_label.mean())
        if len(precision_by_label)
        else 0.0,
        "recall_macro": float(recall_by_label.mean())
        if len(recall_by_label)
        else 0.0,
        "f1_macro": float(f1_by_label.mean())
        if len(f1_by_label)
        else 0.0,
    }


def calculate_matrix_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    if y_true.shape != y_pred.shape:
        raise RuntimeError(
            f"Metric matrices differ in shape: {y_true.shape} vs {y_pred.shape}."
        )

    tp = ((y_true == 1) & (y_pred == 1)).sum(axis=0)
    fp = ((y_true == 0) & (y_pred == 1)).sum(axis=0)
    fn = ((y_true == 1) & (y_pred == 0)).sum(axis=0)

    return _metrics_from_confusion_counts(
        tp,
        fp,
        fn,
    )


def calculate_per_feature_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: Sequence[str],
) -> Dict[str, Dict[str, float]]:
    results: Dict[str, Dict[str, float]] = {}

    for label_id, feature in enumerate(labels):
        true_col = y_true[:, label_id]
        pred_col = y_pred[:, label_id]

        tp = int(np.sum((true_col == 1) & (pred_col == 1)))
        fp = int(np.sum((true_col == 0) & (pred_col == 1)))
        fn = int(np.sum((true_col == 1) & (pred_col == 0)))

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0

        f1 = (
            2.0 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        results[feature] = {
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "support": int(true_col.sum()),
        }

    return results


def print_per_feature_table(
    title: str,
    labels: Sequence[str],
    results: Dict[str, Dict[str, float]],
) -> None:
    print()
    print("=" * 92)
    print(title)
    print("=" * 92)

    heading = (
        f"{'Feature':35s} "
        f"{'Precision':>10s} "
        f"{'Recall':>10s} "
        f"{'F1':>10s} "
        f"{'Support':>10s}"
    )

    print(heading)
    print("-" * len(heading))

    for feature in labels:
        result = results[feature]

        print(
            f"{feature:35s} "
            f"{result['precision']:10.4f} "
            f"{result['recall']:10.4f} "
            f"{result['f1']:10.4f} "
            f"{int(result['support']):10d}"
        )


def build_evaluation_arrays(
    gold_records: Dict[Tuple[int, int], Dict[str, Any]],
    raw_records: Dict[Tuple[int, int], Dict[str, Any]],
    thresholds: Dict[str, float],
    full_labels: Sequence[str],
    independent_labels: Sequence[str],
    allowed_sentence_indices: Optional[Sequence[int]] = None,
) -> Dict[str, Any]:
    """
    Build aligned full and canonical-independent matrices.

    The complete hierarchy-expanded matrices are the headline evaluation.
    Canonical independent matrices are retained as a diagnostic comparable in
    spirit to RoBERTa's direct-output metrics.
    """

    allowed = (
        set(int(x) for x in allowed_sentence_indices)
        if allowed_sentence_indices is not None
        else None
    )

    keys: List[Tuple[int, int]] = []

    for key, gold_record in gold_records.items():
        sentence_index, _ = key

        if allowed is not None and sentence_index not in allowed:
            continue

        gold_features = clean_feature_list(
            gold_record.get("features", [])
        )

        if EXCLUDE_EMPTY_GOLD_RECORDS and not gold_features:
            continue

        if key not in raw_records:
            continue

        keys.append(key)

    keys.sort()

    if not keys:
        raise RuntimeError(
            "No aligned eligible tokens were available for evaluation."
        )

    full_label_set = set(full_labels)

    gold_full_sets: List[Set[str]] = []
    pred_full_sets: List[Set[str]] = []
    gold_independent_sets: List[Set[str]] = []
    pred_independent_sets: List[Set[str]] = []

    for key in keys:
        gold_features = clean_feature_list(
            gold_records[key].get("features", [])
        )
        gold_complete = feature_closure(gold_features)

        unknown_gold = gold_complete - full_label_set

        if unknown_gold:
            raise RuntimeError(
                "Held-out evaluation contains enabled full labels unseen in "
                f"training: {sorted(unknown_gold)}"
            )

        scores = raw_records[key].get("feature_scores", {})

        if not isinstance(scores, dict):
            scores = {}

        direct_pred = select_direct_features(
            scores,
            thresholds,
        )
        pred_complete = feature_closure(direct_pred)

        gold_independent = reduce_to_independent_features(
            gold_complete
        )
        pred_independent = reduce_to_independent_features(
            pred_complete
        )

        gold_full_sets.append(gold_complete)
        pred_full_sets.append(pred_complete)
        gold_independent_sets.append(gold_independent)
        pred_independent_sets.append(pred_independent)

    y_true_full = _binary_matrix_from_sets(
        gold_full_sets,
        full_labels,
    )
    y_pred_full = _binary_matrix_from_sets(
        pred_full_sets,
        full_labels,
    )

    y_true_independent = _binary_matrix_from_sets(
        gold_independent_sets,
        independent_labels,
    )
    y_pred_independent = _binary_matrix_from_sets(
        pred_independent_sets,
        independent_labels,
    )

    return {
        "keys": keys,
        "y_true_full": y_true_full,
        "y_pred_full": y_pred_full,
        "y_true_independent": y_true_independent,
        "y_pred_independent": y_pred_independent,
    }


def build_sentence_slices(
    keys: Sequence[Tuple[int, int]],
) -> Tuple[List[int], List[slice]]:
    """
    Return contiguous token-row slices for sentence-cluster bootstrapping.
    """

    sentence_indices: List[int] = []
    slices: List[slice] = []

    if not keys:
        return sentence_indices, slices

    current_sentence = int(keys[0][0])
    start = 0

    for row, key in enumerate(keys):
        sentence_index = int(key[0])

        if sentence_index != current_sentence:
            sentence_indices.append(current_sentence)
            slices.append(slice(start, row))
            current_sentence = sentence_index
            start = row

    sentence_indices.append(current_sentence)
    slices.append(slice(start, len(keys)))

    return sentence_indices, slices


def bootstrap_test_metrics(
    y_true_full: np.ndarray,
    y_pred_full: np.ndarray,
    sentence_slices: Sequence[slice],
) -> Dict[str, Any]:
    """
    Percentile sentence-cluster bootstrap for one deterministic GLiNER baseline.

    All eligible NOUN/PROPN rows belonging to a sampled sentence move together.
    """

    if TEST_BOOTSTRAP_SAMPLES < 1:
        raise ValueError("TEST_BOOTSTRAP_SAMPLES must be at least 1.")

    if not 0.0 < TEST_BOOTSTRAP_CONFIDENCE < 1.0:
        raise ValueError(
            "TEST_BOOTSTRAP_CONFIDENCE must be strictly between 0 and 1."
        )

    if y_true_full.shape != y_pred_full.shape:
        raise RuntimeError(
            "Bootstrap truth/prediction matrices have different shapes."
        )

    n_sentences = len(sentence_slices)

    if n_sentences == 0:
        raise ValueError("No test sentences are available for bootstrap.")

    n_labels = y_true_full.shape[1]

    tp_sentence = np.zeros(
        (n_sentences, n_labels),
        dtype=np.int64,
    )
    fp_sentence = np.zeros_like(tp_sentence)
    fn_sentence = np.zeros_like(tp_sentence)

    for sentence_pos, row_slice in enumerate(sentence_slices):
        truth = y_true_full[row_slice]
        pred = y_pred_full[row_slice]

        tp_sentence[sentence_pos] = (
            (truth == 1) & (pred == 1)
        ).sum(axis=0)
        fp_sentence[sentence_pos] = (
            (truth == 0) & (pred == 1)
        ).sum(axis=0)
        fn_sentence[sentence_pos] = (
            (truth == 1) & (pred == 0)
        ).sum(axis=0)

    metric_names = [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]

    values = {
        metric: np.empty(TEST_BOOTSTRAP_SAMPLES, dtype=np.float64)
        for metric in metric_names
    }

    rng = np.random.default_rng(TEST_BOOTSTRAP_SEED)

    for bootstrap_index in range(TEST_BOOTSTRAP_SAMPLES):
        sampled_sentences = rng.integers(
            0,
            n_sentences,
            size=n_sentences,
        )

        multiplicities = np.bincount(
            sampled_sentences,
            minlength=n_sentences,
        ).astype(np.int32)

        tp = np.tensordot(
            multiplicities,
            tp_sentence,
            axes=(0, 0),
        )
        fp = np.tensordot(
            multiplicities,
            fp_sentence,
            axes=(0, 0),
        )
        fn = np.tensordot(
            multiplicities,
            fn_sentence,
            axes=(0, 0),
        )

        metrics = _metrics_from_confusion_counts(
            tp,
            fp,
            fn,
        )

        for metric in metric_names:
            values[metric][bootstrap_index] = metrics[metric]

    alpha = 1.0 - TEST_BOOTSTRAP_CONFIDENCE
    lower_q = alpha / 2.0
    upper_q = 1.0 - lower_q

    intervals: Dict[str, Dict[str, float]] = {}

    for metric in metric_names:
        metric_values = values[metric]

        intervals[metric] = {
            "bootstrap_mean": float(metric_values.mean()),
            "bootstrap_sd": float(metric_values.std(ddof=1))
            if len(metric_values) > 1
            else 0.0,
            "ci_lower": float(np.quantile(metric_values, lower_q)),
            "ci_upper": float(np.quantile(metric_values, upper_q)),
            "confidence": float(TEST_BOOTSTRAP_CONFIDENCE),
            "bootstrap_samples": int(TEST_BOOTSTRAP_SAMPLES),
        }

    return {
        "bootstrap_unit": "sentence",
        "statistic": "single_deterministic_baseline_metric",
        "seed": TEST_BOOTSTRAP_SEED,
        "n_test_sentences": n_sentences,
        "intervals": intervals,
    }


def print_metric_block(
    title: str,
    metrics: Dict[str, float],
) -> None:
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)

    for metric in [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]:
        print(f"{metric:25s}: {metrics[metric]:.6f}")


def print_bootstrap_table(
    point_metrics: Dict[str, float],
    bootstrap_result: Dict[str, Any],
) -> None:
    print()
    print("=" * 100)
    print("SENTENCE-CLUSTER BOOTSTRAP ON HELD-OUT TEST SET")
    print("=" * 100)
    print(f"Bootstrap samples: {TEST_BOOTSTRAP_SAMPLES:,}")
    print(
        f"Confidence level : {100.0 * TEST_BOOTSTRAP_CONFIDENCE:.1f}%"
    )
    print(
        "Bootstrap unit   : sentence "
        "(all eligible NOUN/PROPN targets from a sampled sentence move together)"
    )
    print()
    print(
        f"{'Metric':25s} "
        f"{'Point estimate':>14s} "
        f"{'95% CI low':>12s} "
        f"{'95% CI high':>12s}"
    )
    print("-" * 70)

    intervals = bootstrap_result["intervals"]

    for metric in [
        "precision_micro",
        "recall_micro",
        "f1_micro",
        "precision_macro",
        "recall_macro",
        "f1_macro",
    ]:
        interval = intervals[metric]

        print(
            f"{metric:25s} "
            f"{point_metrics[metric]:14.4f} "
            f"{interval['ci_lower']:12.4f} "
            f"{interval['ci_upper']:12.4f}"
        )


# =============================================================================
# Apply frozen thresholds to raw test scores
# =============================================================================

def write_thresholded_output(
    raw_path: Path,
    output_path: Path,
    thresholds: Dict[str, float],
) -> None:
    """Convert raw test scores into final direct + inherited feature tags."""

    raw_records = load_jsonl(raw_path)
    tagged_count = 0

    with output_path.open("w", encoding="utf-8") as outfile:
        for record in raw_records:
            scores = record.get("feature_scores", {})

            if not isinstance(scores, dict):
                scores = {}

            direct_features = select_direct_features(
                scores,
                thresholds,
            )
            full_features = feature_closure(
                direct_features
            )

            output: Dict[str, Any] = {
                "sentence_index": record.get("sentence_index"),
                "doc_id": record.get("doc_id"),
                "par_id": record.get("par_id"),
                "sent_id": record.get("sent_id"),
                "token_id": record.get("token_id"),
                "form": record.get("form"),
                "lemma": record.get("lemma"),
                "upos": record.get("upos"),
                "sentence": record.get("sentence"),
                "direct_features": sorted(direct_features),
                "features": sorted(full_features),
                "untagged": len(direct_features) == 0,
                "feature_scores": {
                    feature: float(scores.get(feature, 0.0))
                    for feature in EXTRACTION_FEATURES
                },
                "thresholds": {
                    feature: float(
                        thresholds.get(feature, DEFAULT_FINAL_THRESHOLD)
                    )
                    for feature in EXTRACTION_FEATURES
                },
            }

            if SAVE_MATCHED_ENTITIES and "matched_entities" in record:
                output["matched_entities"] = record["matched_entities"]

            if direct_features:
                tagged_count += 1

            outfile.write(
                json.dumps(output, ensure_ascii=False)
                + "\n"
            )

    print()
    print("=" * 100)
    print("FROZEN-THRESHOLD TEST OUTPUT")
    print("=" * 100)
    print(f"Tokens written               : {len(raw_records):,}")
    print(f"Tokens with >=1 direct tag   : {tagged_count:,}")
    print(f"Output                       : {output_path.resolve()}")


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)

    output_paths = [
        GLINER_SPLIT_PATH,
        THRESHOLDS_PATH,
        TRAIN_RAW_SCORES_PATH,
        TEST_RAW_SCORES_PATH,
        TEST_OUTPUT_PATH,
        TRAIN_UNMATCHED_SPANS_PATH,
        TEST_UNMATCHED_SPANS_PATH,
        TEST_RESULTS_PATH,
        BOOTSTRAP_RESULTS_PATH,
    ]

    for path in output_paths:
        path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    print("=" * 100)
    print("GLiNER2.5 ZERO-SHOT BASELINE — RoBERTa-ALIGNED EVALUATION")
    print("=" * 100)

    disabled_features = sorted(
        feature
        for feature, enabled in FEATURE_SWITCHES.items()
        if not enabled
    )

    print()
    print(f"Model                    : {MODEL_NAME}")
    print(f"Direct extraction labels : {len(EXTRACTION_FEATURES)}")
    print(f"Raw extraction threshold : {MODEL_OUTPUT_THRESHOLD:.4f}")
    print(f"Chunk size               : {LONG_CHUNK_SIZE}")
    print(f"Chunk overlap            : {LONG_CHUNK_OVERLAP}")
    print(f"Overlap policy           : {OVERLAP_POLICY}")
    print(
        "Disabled features        : "
        + (
            ", ".join(disabled_features)
            if disabled_features
            else "(none)"
        )
    )
    print(f"Canonical split path      : {SHARED_SPLIT_PATH}")

    if "Web_location" in FEATURE_DESCRIPTIONS:
        raise RuntimeError(
            "Feature naming error: use 'Web_Location', not 'Web_location'."
        )

    if (
        feature_is_enabled("Web_Location")
        and "Web_Location" not in FEATURE_DESCRIPTIONS
    ):
        raise RuntimeError(
            "Missing required direct feature 'Web_Location'."
        )

    # -------------------------------------------------------------------------
    # Load TRAIN gold and exact sentence-level validation split.
    # -------------------------------------------------------------------------

    gold_train_records = load_gold_records(
        GOLD_TRAIN_JSONL_PATH
    )

    split = load_or_create_validation_split(
        gold_train_records
    )

    full_labels, independent_labels = build_gold_label_vocabularies(
        gold_train_records,
        split["train"],
    )

    print()
    print("Evaluation vocabularies derived from TRAIN subset only:")
    print(f"  Complete hierarchy features : {len(full_labels):,}")
    print(f"  Independent features        : {len(independent_labels):,}")

    # -------------------------------------------------------------------------
    # Device + model.
    # -------------------------------------------------------------------------

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print()
    print(f"Device: {device}")
    print(f"Loading GLiNER2.5 model: {MODEL_NAME}")

    model = AutoExtractor.from_pretrained(
        MODEL_NAME,
        map_location=device,
    )

    # -------------------------------------------------------------------------
    # 1. Raw TRAINING extraction.
    # -------------------------------------------------------------------------

    extract_raw_scores_for_corpus(
        model=model,
        conllu_path=TRAIN_CONLLU_PATH,
        output_path=TRAIN_RAW_SCORES_PATH,
        unmatched_path=TRAIN_UNMATCHED_SPANS_PATH,
        corpus_name="training",
    )

    training_raw_records = load_raw_score_records(
        TRAIN_RAW_SCORES_PATH
    )

    # -------------------------------------------------------------------------
    # 2. Validation-only per-feature threshold optimisation.
    # -------------------------------------------------------------------------

    (
        thresholds,
        diagnostics,
        validation_keys,
    ) = optimise_thresholds_on_validation(
        gold_records=gold_train_records,
        raw_records=training_raw_records,
        validation_sentence_indices=split["validation"],
    )

    print_threshold_table(
        diagnostics
    )

    validation_arrays = build_evaluation_arrays(
        gold_records=gold_train_records,
        raw_records=training_raw_records,
        thresholds=thresholds,
        full_labels=full_labels,
        independent_labels=independent_labels,
        allowed_sentence_indices=split["validation"],
    )

    validation_metrics_full = calculate_matrix_metrics(
        validation_arrays["y_true_full"],
        validation_arrays["y_pred_full"],
    )

    validation_metrics_independent = calculate_matrix_metrics(
        validation_arrays["y_true_independent"],
        validation_arrays["y_pred_independent"],
    )

    print_metric_block(
        "VALIDATION PERFORMANCE WITH OPTIMISED THRESHOLDS — COMPLETE HIERARCHY",
        validation_metrics_full,
    )

    print_metric_block(
        "VALIDATION PERFORMANCE WITH OPTIMISED THRESHOLDS — INDEPENDENT DIAGNOSTIC",
        validation_metrics_independent,
    )

    threshold_document = {
        "model": MODEL_NAME,
        "method": "per_feature_validation_f1",
        "gold_target_for_threshold_calibration": "complete_feature_membership",
        "validation_source": "training_corpus_only",
        "validation_split_path": str(SHARED_SPLIT_PATH),
        "raw_extraction_threshold": MODEL_OUTPUT_THRESHOLD,
        "raw_cache_config": current_raw_cache_config(),
        "default_final_threshold": DEFAULT_FINAL_THRESHOLD,
        "feature_switches": FEATURE_SWITCHES,
        "disabled_features": disabled_features,
        "extraction_features": EXTRACTION_FEATURES,
        "full_evaluation_labels": full_labels,
        "independent_evaluation_labels": independent_labels,
        "thresholds": {
            feature: float(thresholds[feature])
            for feature in EXTRACTION_FEATURES
        },
        "diagnostics": diagnostics,
        "validation_metrics_complete": validation_metrics_full,
        "validation_metrics_independent": validation_metrics_independent,
    }

    with THRESHOLDS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            threshold_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print()
    print(
        "Frozen thresholds saved to:\n"
        f"  {THRESHOLDS_PATH.resolve()}"
    )

    # At this point all threshold/model-selection decisions are frozen.
    frozen_thresholds = dict(thresholds)

    # -------------------------------------------------------------------------
    # 3. Raw official TEST extraction.
    # -------------------------------------------------------------------------

    extract_raw_scores_for_corpus(
        model=model,
        conllu_path=TEST_CONLLU_PATH,
        output_path=TEST_RAW_SCORES_PATH,
        unmatched_path=TEST_UNMATCHED_SPANS_PATH,
        corpus_name="test",
    )

    test_raw_records = load_raw_score_records(
        TEST_RAW_SCORES_PATH
    )

    # -------------------------------------------------------------------------
    # 4. Apply frozen validation thresholds to TEST predictions.
    # -------------------------------------------------------------------------

    write_thresholded_output(
        raw_path=TEST_RAW_SCORES_PATH,
        output_path=TEST_OUTPUT_PATH,
        thresholds=frozen_thresholds,
    )

    # -------------------------------------------------------------------------
    # 5. FINAL TEST EVALUATION.
    #
    # Gold test labels are first read here, after thresholds are frozen.
    # -------------------------------------------------------------------------

    gold_test_records = load_gold_records(
        GOLD_TEST_JSONL_PATH
    )

    test_arrays = build_evaluation_arrays(
        gold_records=gold_test_records,
        raw_records=test_raw_records,
        thresholds=frozen_thresholds,
        full_labels=full_labels,
        independent_labels=independent_labels,
    )

    test_metrics_full = calculate_matrix_metrics(
        test_arrays["y_true_full"],
        test_arrays["y_pred_full"],
    )

    test_metrics_independent = calculate_matrix_metrics(
        test_arrays["y_true_independent"],
        test_arrays["y_pred_independent"],
    )

    test_sentence_indices, sentence_slices = build_sentence_slices(
        test_arrays["keys"]
    )

    print()
    print("Held-out test evaluation set:")
    print(f"  Eligible sentences : {len(test_sentence_indices):,}")
    print(f"  Eligible tokens    : {len(test_arrays['keys']):,}")

    # Test-at-0.5 diagnostic, matching the RoBERTa report.
    default_thresholds = {
        feature: float(DEFAULT_FINAL_THRESHOLD)
        for feature in EXTRACTION_FEATURES
    }

    test_default_arrays = build_evaluation_arrays(
        gold_records=gold_test_records,
        raw_records=test_raw_records,
        thresholds=default_thresholds,
        full_labels=full_labels,
        independent_labels=independent_labels,
    )

    test_default_metrics_full = calculate_matrix_metrics(
        test_default_arrays["y_true_full"],
        test_default_arrays["y_pred_full"],
    )

    test_default_metrics_independent = calculate_matrix_metrics(
        test_default_arrays["y_true_independent"],
        test_default_arrays["y_pred_independent"],
    )

    print_metric_block(
        "TEST PERFORMANCE AT DEFAULT 0.5 — COMPLETE HIERARCHY",
        test_default_metrics_full,
    )

    print_metric_block(
        "TEST PERFORMANCE WITH VALIDATION-OPTIMISED THRESHOLDS — COMPLETE HIERARCHY",
        test_metrics_full,
    )

    print_metric_block(
        "TEST PERFORMANCE WITH VALIDATION-OPTIMISED THRESHOLDS — INDEPENDENT DIAGNOSTIC",
        test_metrics_independent,
    )

    # -------------------------------------------------------------------------
    # Per-feature test tables.
    # -------------------------------------------------------------------------

    per_feature_complete = calculate_per_feature_metrics(
        y_true=test_arrays["y_true_full"],
        y_pred=test_arrays["y_pred_full"],
        labels=full_labels,
    )

    per_feature_independent = calculate_per_feature_metrics(
        y_true=test_arrays["y_true_independent"],
        y_pred=test_arrays["y_pred_independent"],
        labels=independent_labels,
    )

    print_per_feature_table(
        "TEST PERFORMANCE BY COMPLETE FEATURE (AFTER INHERITANCE)",
        full_labels,
        per_feature_complete,
    )

    print_per_feature_table(
        "TEST PERFORMANCE BY CANONICAL INDEPENDENT FEATURE",
        independent_labels,
        per_feature_independent,
    )

    # -------------------------------------------------------------------------
    # Sentence-cluster bootstrap on primary hierarchy-expanded test metrics.
    # -------------------------------------------------------------------------

    bootstrap_result = bootstrap_test_metrics(
        y_true_full=test_arrays["y_true_full"],
        y_pred_full=test_arrays["y_pred_full"],
        sentence_slices=sentence_slices,
    )

    print_bootstrap_table(
        point_metrics=test_metrics_full,
        bootstrap_result=bootstrap_result,
    )

    # -------------------------------------------------------------------------
    # Save final test + bootstrap results.
    # -------------------------------------------------------------------------

    result_document = {
        "model": MODEL_NAME,
        "baseline_type": "zero_shot_gliner2.5_span_extraction",
        "stochastic_repeated_runs": False,
        "repeated_run_note": (
            "No repeated training runs are performed because GLiNER weights are "
            "not fine-tuned. Test-sample uncertainty is estimated with the same "
            "sentence-cluster bootstrap design used for the RoBERTa experiment."
        ),
        "feature_switches": FEATURE_SWITCHES,
        "disabled_features": disabled_features,
        "validation_split": {
            "source": str(SHARED_SPLIT_PATH),
            "train_sentences": len(split["train"]),
            "validation_sentences": len(split["validation"]),
        },
        "test_size": {
            "eligible_sentences": len(test_sentence_indices),
            "eligible_tokens": len(test_arrays["keys"]),
        },
        "full_labels": full_labels,
        "independent_labels": independent_labels,
        "thresholds_optimised_on": "validation_only",
        "thresholds": frozen_thresholds,
        "test_metrics_default_0_5_complete": test_default_metrics_full,
        "test_metrics_default_0_5_independent": test_default_metrics_independent,
        "test_metrics_optimised_complete": test_metrics_full,
        "test_metrics_optimised_independent": test_metrics_independent,
        "per_feature_complete": per_feature_complete,
        "per_feature_independent": per_feature_independent,
        "bootstrap": bootstrap_result,
    }

    with TEST_RESULTS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            result_document,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    with BOOTSTRAP_RESULTS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(
            bootstrap_result,
            handle,
            ensure_ascii=False,
            indent=2,
        )

    print()
    print("=" * 100)
    print("FINISHED")
    print("=" * 100)
    print()
    print(
        "The official test gold annotations were used only after validation "
        "threshold calibration had been frozen. The canonical train/validation "
        "split is model-independent and validated against the enabled-feature "
        "eligible sentence set."
    )
    print()
    print(f"Thresholds       : {THRESHOLDS_PATH.resolve()}")
    print(f"Test output      : {TEST_OUTPUT_PATH.resolve()}")
    print(f"Test results     : {TEST_RESULTS_PATH.resolve()}")
    print(f"Bootstrap results: {BOOTSTRAP_RESULTS_PATH.resolve()}")


if __name__ == "__main__":
    main()


GLiNER2.5 ZERO-SHOT BASELINE — RoBERTa-ALIGNED EVALUATION

Model                    : fastino/gliner2.5-base-v1
Direct extraction labels : 39
Raw extraction threshold : 0.0010
Chunk size               : 48
Chunk overlap            : 8
Overlap policy           : allow
Disabled features        : Rigidity, Shape, State, State_of_Matter
Canonical split path      : ../data/semantic_feature_sentence_split.json

Sentence-level split inventory:
  Eligible sentences          : 11,198
  Expected validation (20%)  : 2,240
  Expected training           : 8,958

Using validated canonical shared sentence split.
Training sentences   : 8,885
Validation sentences : 2,313

Evaluation vocabularies derived from TRAIN subset only:
  Complete hierarchy features : 39
  Independent features        : 37

Device: cuda
Loading GLiNER2.5 model: fastino/gliner2.5-base-v1


[transformers] You are using a model of type `gliner2` to instantiate a model of type `extractor`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
[transformers] You are using a model of type `gliner2` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
/root/wiktextract/.venv/lib/python3.10/site-packages/gliner2/models/boundary/model.py:1147: RuntimeWarning: Encoder rejected attn_implementation='sdpa'; falling back to 'eager' (DebertaV2Model does not support an attention im


RAW GLiNER EXTRACTION — TRAINING
Sentences       : 12,544
Documents       : 540
NOUN/PROPN      : 47,371
Feature classes : 39


training documents: 100%|█████████████████████████████████████████████████████████████| 540/540 [09:42<00:00,  1.08s/it]



Raw extracted spans              : 1,843,113
Raw unmatched spans              : 416,711
Sentence alignment fallbacks     : 0
Raw token-score output           : /home/mark/verbnet-roberta/models/gliner2.5/ewt_train_gliner2_raw_scores.jsonl

VALIDATION-OPTIMISED GLiNER THRESHOLDS
Feature                              Threshold   Support     F1@0.5     F1 opt     Change
-----------------------------------------------------------------------------------------
Abstract                                0.0011      4918     0.1888     0.6986    +0.5098
Animal                                  0.3136       184     0.7102     0.7316    +0.0214
Animate                                 0.0010      1838     0.0858     0.2968    +0.2111
Artefact                                0.0211       800     0.2876     0.3784    +0.0908
Body_Part                               0.5483        74     0.6380     0.6420    +0.0039
Comestible                              0.0329       157     0.6061     0.7166    +0.1105


test documents: 100%|█████████████████████████████████████████████████████████████████| 316/316 [01:08<00:00,  4.63it/s]



Raw extracted spans              : 249,487
Raw unmatched spans              : 57,700
Sentence alignment fallbacks     : 0
Raw token-score output           : /home/mark/verbnet-roberta/models/gliner2.5/wt_test_gliner2_raw_scores.jsonl

FROZEN-THRESHOLD TEST OUTPUT
Tokens written               : 6,198
Tokens with >=1 direct tag   : 6,042
Output                       : /home/mark/verbnet-roberta/models/gliner2.5/ewt_test_gliner2_noun_features_span.jsonl

Held-out test evaluation set:
  Eligible sentences : 1,759
  Eligible tokens    : 5,704

TEST PERFORMANCE AT DEFAULT 0.5 — COMPLETE HIERARCHY
precision_micro          : 0.793395
recall_micro             : 0.351566
f1_micro                 : 0.487231
precision_macro          : 0.584959
recall_macro             : 0.316844
f1_macro                 : 0.346291

TEST PERFORMANCE WITH VALIDATION-OPTIMISED THRESHOLDS — COMPLETE HIERARCHY
precision_micro          : 0.541093
recall_micro             : 0.588341
f1_micro                 : 0.563729
p

<h1>Inspection</h1>

In [3]:
import csv
import random
import textwrap

from pathlib import Path


# =============================================================================
# Configuration
# =============================================================================

# True: show only nouns where GLiNER disagrees with gold.
# False: show correct and incorrect predictions.
ONLY_MISMATCHES = True

# Maximum number of examples to print.
# Set to None to print everything.
MAX_EXAMPLES = 100

# True: randomly sample examples.
# False: show examples in their original test-set order.
RANDOM_SAMPLE = True
SAMPLE_SEED = 42

# Include the sentence containing each noun.
SHOW_SENTENCE = True

# Save ALL aligned comparisons, regardless of printing filters.
COMPARISON_CSV_PATH = TEST_OUTPUT_PATH.with_name(
    "ewt_test_gliner2_gold_comparison.csv"
)


# =============================================================================
# Build comparison records
# =============================================================================

gold_records = load_gold_records(
    GOLD_TEST_JSONL_PATH
)

gliner_records = load_jsonl(
    TEST_OUTPUT_PATH
)

gliner_by_key = {}

for record in gliner_records:
    if record.get("upos") not in TARGET_POS:
        continue

    key = (
        int(record["sentence_index"]),
        int(record["token_id"]),
    )

    if key in gliner_by_key:
        raise ValueError(f"Duplicate GLiNER token key: {key}")

    gliner_by_key[key] = record


comparisons = []
missing_predictions = []
excluded_empty_gold = 0

for key in sorted(gold_records):

    gold_record = gold_records[key]

    # Match the original evaluation's treatment of disabled features.
    gold_features = clean_feature_list(
        gold_record.get("features", [])
    )

    # Match the original evaluation's eligible-token filtering.
    if EXCLUDE_EMPTY_GOLD_RECORDS and not gold_features:
        excluded_empty_gold += 1
        continue

    gliner_record = gliner_by_key.get(key)

    if gliner_record is None:
        missing_predictions.append(key)
        continue

    # Complete gold hierarchy, exactly as in the evaluation.
    gold_complete = feature_closure(gold_features)

    # The output's "features" field already contains inherited features.
    gliner_complete = clean_feature_list(
        gliner_record.get("features", [])
    )

    correct = gold_complete & gliner_complete
    missing = gold_complete - gliner_complete
    extra = gliner_complete - gold_complete

    comparisons.append({
        "sentence_index": key[0],
        "token_id": key[1],
        "sent_id": gliner_record.get("sent_id"),
        "form": gliner_record.get("form", ""),
        "lemma": gliner_record.get("lemma", ""),
        "upos": gliner_record.get("upos", ""),
        "sentence": gliner_record.get("sentence", ""),

        "gold": sorted(gold_complete),
        "gliner": sorted(gliner_complete),
        "direct": sorted(
            gliner_record.get("direct_features", [])
        ),

        "correct": sorted(correct),
        "missing": sorted(missing),
        "extra": sorted(extra),

        "exact_match": gold_complete == gliner_complete,
    })


# A missing prediction should not silently disappear from the comparison.
if missing_predictions:
    raise RuntimeError(
        f"{len(missing_predictions)} eligible gold tokens have no "
        f"matching GLiNER output. First missing keys: "
        f"{missing_predictions[:10]}"
    )


# =============================================================================
# Export all comparisons to CSV
# =============================================================================

COMPARISON_CSV_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

fieldnames = [
    "sentence_index",
    "token_id",
    "sent_id",
    "form",
    "lemma",
    "upos",
    "sentence",
    "gold",
    "gliner",
    "direct",
    "correct",
    "missing",
    "extra",
    "exact_match",
]

with COMPARISON_CSV_PATH.open(
    "w",
    encoding="utf-8-sig",
    newline="",
) as outfile:

    writer = csv.DictWriter(
        outfile,
        fieldnames=fieldnames,
    )
    writer.writeheader()

    for record in comparisons:
        output = dict(record)

        for field in [
            "gold",
            "gliner",
            "direct",
            "correct",
            "missing",
            "extra",
        ]:
            output[field] = ", ".join(output[field])

        writer.writerow(output)


# =============================================================================
# Summary
# =============================================================================

exact_matches = sum(
    r["exact_match"]
    for r in comparisons
)

mismatches = len(comparisons) - exact_matches

print("=" * 120)
print("GLiNER2.5 VS GOLD — TEST SET COMPARISON")
print("=" * 120)

print(f"Gold NOUN/PROPN records       : {len(gold_records):,}")
print(f"Excluded empty gold records  : {excluded_empty_gold:,}")
print(f"Evaluated tokens             : {len(comparisons):,}")
print(f"Exact feature-set matches    : {exact_matches:,}")
print(f"Feature-set mismatches       : {mismatches:,}")

if comparisons:
    print(
        f"Exact match percentage       : "
        f"{100 * exact_matches / len(comparisons):.2f}%"
    )

print(f"\nComplete CSV: {COMPARISON_CSV_PATH.resolve()}")


# =============================================================================
# Select examples
# =============================================================================

examples = comparisons

if ONLY_MISMATCHES:
    examples = [
        r for r in examples
        if not r["exact_match"]
    ]

if MAX_EXAMPLES is not None:
    if RANDOM_SAMPLE:
        rng = random.Random(SAMPLE_SEED)
        examples = rng.sample(
            examples,
            min(MAX_EXAMPLES, len(examples)),
        )
    else:
        examples = examples[:MAX_EXAMPLES]

# Restore corpus order for convenient contextual reading.
examples = sorted(
    examples,
    key=lambda r: (
        r["sentence_index"],
        r["token_id"],
    ),
)


# =============================================================================
# Print side-by-side feature comparisons
# =============================================================================

COLUMN_WIDTH = 65


def wrap_features(features, width=COLUMN_WIDTH):
    value = ", ".join(features) if features else "(none)"

    return textwrap.wrap(
        value,
        width=width,
        break_long_words=False,
        break_on_hyphens=False,
    ) or ["(none)"]


print()
print("=" * 140)
print(f"INDIVIDUAL EXAMPLES ({len(examples):,})")
print("=" * 140)

for number, record in enumerate(examples, start=1):

    print()
    print("-" * 140)

    print(
        f"[{number}] {record['form']!r} "
        f"({record['upos']})"
        f"  |  Sentence {record['sentence_index']}"
        f"  |  Token {record['token_id']}"
    )

    if SHOW_SENTENCE:
        print(f"\nSentence: {record['sentence']}")

    print()
    print(
        f"{'GOLD FEATURES (INHERITED)':<{COLUMN_WIDTH}}"
        " | "
        "GLiNER2.5 FEATURES (INHERITED)"
    )
    print("-" * 140)

    gold_lines = wrap_features(record["gold"])
    gliner_lines = wrap_features(record["gliner"])

    line_count = max(
        len(gold_lines),
        len(gliner_lines),
    )

    for i in range(line_count):

        gold_line = (
            gold_lines[i]
            if i < len(gold_lines)
            else ""
        )

        gliner_line = (
            gliner_lines[i]
            if i < len(gliner_lines)
            else ""
        )

        print(
            f"{gold_line:<{COLUMN_WIDTH}}"
            f" | {gliner_line}"
        )

    print()
    print(
        "Missing from GLiNER : "
        + (
            ", ".join(record["missing"])
            if record["missing"]
            else "(none)"
        )
    )

    print(
        "Extra from GLiNER   : "
        + (
            ", ".join(record["extra"])
            if record["extra"]
            else "(none)"
        )
    )

    print(
        "Exact match         : "
        + ("YES" if record["exact_match"] else "NO")
    )

print()
print("=" * 140)
print("COMPARISON COMPLETE")
print("=" * 140)

GLiNER2.5 VS GOLD — TEST SET COMPARISON
Gold NOUN/PROPN records       : 6,198
Excluded empty gold records  : 494
Evaluated tokens             : 5,704
Exact feature-set matches    : 674
Feature-set mismatches       : 5,030
Exact match percentage       : 11.82%

Complete CSV: /home/mark/verbnet-roberta/models/gliner2.5/ewt_test_gliner2_gold_comparison.csv

INDIVIDUAL EXAMPLES (100)

--------------------------------------------------------------------------------------------------------------------------------------------
[1] 'stuff' (NOUN)  |  Sentence 21  |  Token 27

Sentence: (You don't need to use their site, you can opt-out of sharing your information, you don't need to send stuff to anyone with a Gmail account, and if -- wonder of wonders -- you're worried that you might send something to someone who would forward an excerpt to someone who would then store it on a Gmail account... you have far, far too much time on your hands).

GOLD FEATURES (INHERITED)                            